In [ ]:
#CELL 1

#install the 2 new libraries needed for today
#chromadb:vecto database (like SQLite but for AI embeddings)
#sentence-transformers: converts text to 384- dimentional numbers

!pip install chromadb sentence-transformers -q
#-q means quiet mode -- reduces the amount of output printed
#you will see a progress bar. Wait until it says 'successfully insalled'
print('Installation completed')


Installation completed


In [ ]:
#CELL 2
#standard libraries we already know from earlier days
import pandas as pd            # For loading and exploring the CSV dataset
import numpy as np             # For working with numerical arrays (embedding vectors)

#New library : sentence-transformers
#sentencetransformers: the class we use to load the embedding model
from sentence_transformers import SentenceTransformer

# New library : chromaDb
# chromadB: the vector database package
import chromadb

print("All libraries imported successfully")
print(f"ChromaDB version : {chromadb.__version__}" )


All libraries imported successfully
ChromaDB version : 1.5.9


In [ ]:
#CELL 3
# DEMONSTRATION - KEYWORD SEARCH vs SEMANTIC SEARCH
#Imagine we have a small collection of documents about data
documents = [
    "ETL is used to clean and tranform data",                       #doc_0
    "A vehicle is a mode of transportation",                        #doc_1
    "Cars and trucks are popular automobiles",                      #doc_2
    "SQL is used to query database",                                #doc_3
    "Machine learning trains models on data",                       #doc_4
]

#--------------KEYWORD SEARCH----------------
#keyword search : check id the exact query word apperas in the document

query_keyword = "vehicle"             #the word we are searching for

print("="*60)
print(f"KEYWORD SEARCH for: '{query_keyword}")
print("="*60)
for i, doc in enumerate(documents):
  #.lower() marks comparison case-sensitive
  #'in' checks if the word appears anywhere in the string
  if query_keyword.lower() in doc.lower():
    print(f"   FOUND [doc_{i}]: {doc}")
  else:
    print(f"  MISSED [doc_{i}]: {doc}")

print()
print("PROBLEM: doc_2 talks about 'Cars and trucks' - which ARE vehicles!" )
print("But keyword sarch MISSED it because it searched for the exact word 'vehicle'.")

KEYWORD SEARCH for: 'vehicle
  MISSED [doc_0]: ETL is used to clean and tranform data
   FOUND [doc_1]: A vehicle is a mode of transportation
  MISSED [doc_2]: Cars and trucks are popular automobiles
  MISSED [doc_3]: SQL is used to query database
  MISSED [doc_4]: Machine learning trains models on data

PROBLEM: doc_2 talks about 'Cars and trucks' - which ARE vehicles!
But keyword sarch MISSED it because it searched for the exact word 'vehicle'.


In [ ]:
#CELL 4
#More examples of keyword search failures
failure_examples = [
    {"query":"I feel sick",       "misses":"I am unwell, patient has fever"},
    {"query":"How to cook rice",  "misses":"Steps to prepare rice"},
    {"query":"vehicle speed",     "misses":"car acceleration,automobile velocity"},
    {"query":"ML model accuracy" , "misses":"classification performance,prediction quality"  }
]

print("KEYWORD SEARCH FAILURE CASES")
print("="*60)
for ex in failure_examples:
  #f-string: embeds variables inside the string using {}
  print(f"Query: '{ex['query']}'")
  print(f"  Misses: '{ex['misses']}'")
  print("-" * 40)

print()
print("SOLUTION: We need search that understans MEANING, not just characters.")
print("That is what EMBEDDINGS do.")

KEYWORD SEARCH FAILURE CASES
Query: 'I feel sick'
  Misses: 'I am unwell, patient has fever'
----------------------------------------
Query: 'How to cook rice'
  Misses: 'Steps to prepare rice'
----------------------------------------
Query: 'vehicle speed'
  Misses: 'car acceleration,automobile velocity'
----------------------------------------
Query: 'ML model accuracy'
  Misses: 'classification performance,prediction quality'
----------------------------------------

SOLUTION: We need search that understans MEANING, not just characters.
That is what EMBEDDINGS do.


# KEY VOCABULARY

| Term | Meaning |
|:---|:---|
| **vector** | A list of numbers (our embedding are list of 384 numbers) |
| **Dimension** |  One number in the list(384 dimensions = 384 numbers) |
| **Cosine Similarity** | A score from 0.0 to 1.0 - how similar two vector are |
|**Embedding Model** | The neural network that converts texts to vectors|

In [ ]:
#CELL 5

#- 'v2' means : version 2
# Output: 384-dimenstional vectors
#note: first run downloads the model(`80MB). This is a on-time download.
print("Loading embedding model...  (may take 1-2 minutes on first run)")

model = SentenceTransformer('all-MiniLM-L6-v2')
#the model is now loaded and ready to convert text to embeddings
print("Model Loaded Successfully")
print(f'Model produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions')

Loading embedding model...  (may take 1-2 minutes on first run)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully
Model produces vectors of size: 384 dimensions


/tmp/ipykernel_875/1202065171.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Model produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions')


In [ ]:
#CELL 6
#Define a single sentence to embed
sentence ="ETL is used to clean and tranform data"

#model.encode() converts to text to a vector(list of number)
#InputL a string OR a list of strings
#Output: NumPy array
embedding = model.encode(sentence)

#Let us explore
print(f"input sentence:  {sentence}")
print()
print(f"Embedding type: {type(embedding)}")
#type shows this is a numpy.ndarray -an array of numbers

print(f"Embedding shape: {embedding.shape}")
#shape(384,) means: 1 sentence * 384 numbers

print(f"First 10 numbers:{embedding[:10].round(4)}")
#these are the first 10 of 384 numbers

print(f"Min value: {embedding.min():.4f}")
print(f"Max value: {embedding.max():.4f}")

input sentence:  ETL is used to clean and tranform data

Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)
First 10 numbers:[-0.0574  0.042   0.0164 -0.0315  0.0129 -0.0885  0.0336 -0.0191  0.0766
  0.0283]
Min value: -0.1425
Max value: 0.1618


In [ ]:
#CELL 7

#each row= one sentence's embedding (384 numbers)
sentences=[
    "ETL is used to clean and transform the data",           #sentence 0
    "Data transformation is a key pipeline step",            #sentence 1 - similar meaning to sentence 0
    "The sky is blue and clouds are white",                  #sentence 2 - completely different from the sentence 0 and sentence 1
]

#Embed all the three sentences at once
embeddings = model.encode(sentences)
#embedddings is a 2D NumPy array

print(f"Number of sentences: {len(sentences)}")
print(f"Embeddings shape: {embeddings.shape}")
#Expected output (3,384)
#3 = number of sentences
#384 = dimension per sentence

print()
print("each row is one sentence's embedding:")
for i, sent in enumerate(sentences):
  # embedding[i] gives row i (sentence i's vector)
  print(f" Sentence {i}:shape={embeddings[i].shape}, first 5 values={embeddings[i][:5].round(3)}")

Number of sentences: 3
Embeddings shape: (3, 384)

each row is one sentence's embedding:
 Sentence 0:shape=(384,), first 5 values=[-0.074  0.049  0.016 -0.034  0.033]
 Sentence 1:shape=(384,), first 5 values=[-0.047  0.059 -0.001 -0.033 -0.046]
 Sentence 2:shape=(384,), first 5 values=[0.054 0.06  0.075 0.049 0.05 ]


In [ ]:
#CELL 8

#Cosine similarity : measures the angle between 2 vectors
#score from 0.0 to 0.1
#we do not need to know the formula for practical use
#Just remember:higher score = more similar

def cosine_similarity(vec_a, vec_b):
  """Calculate cosine similarity between two vectors"""
  #np.dot: dot product of two arrays (multiply element-wise, then sum)
  dot_product=np.dot(vec_a, vec_b)

  #np.linalg.norm: lem=ngth (magnitude) of the vector
  norm_a=np.linalg.norm(vec_a)
  norm_b=np.linalg.norm(vec_b)

  #divide dot product by product of lengths
  return dot_product/(norm_a * norm_b)

#calculate similarities
sim_01 = cosine_similarity(embeddings[0], embeddings[1])
sim_02 = cosine_similarity(embeddings[0], embeddings[2])
sim_12 = cosine_similarity(embeddings[1], embeddings[2])

print(f"COSINE SIMILARITY SCORES")
print("="*60)
print(f"Sentence 0: '{sentences[0]}'")
print(f"Sentence 1: '{sentences[1]}'")
print(f"Sentence 2: '{sentences[2]}'")
print()
print(f"Similarity (0 vs 1): {sim_01:.4f}  <-  Expected ; HIGH (same topic)")
print(f"Similarity (0 vs 2): {sim_02:.4f}  <-  Expected ; LOW (different topic)")
print(f"Similarity (1 vs 2): {sim_12:.4f}  <-  Expected ; LOW (different topic)")
print()
print("INSIGHT: Sentences 0 and 1 have different words but similar meaning")
print("Their cosine similarity score is high - the embedding captured the meaning.")

COSINE SIMILARITY SCORES
Sentence 0: 'ETL is used to clean and transform the data'
Sentence 1: 'Data transformation is a key pipeline step'
Sentence 2: 'The sky is blue and clouds are white'

Similarity (0 vs 1): 0.4304  <-  Expected ; HIGH (same topic)
Similarity (0 vs 2): -0.0110  <-  Expected ; LOW (different topic)
Similarity (1 vs 2): 0.0363  <-  Expected ; LOW (different topic)

INSIGHT: Sentences 0 and 1 have different words but similar meaning
Their cosine similarity score is high - the embedding captured the meaning.


In [ ]:
#CELL 9

#my semtence
your_sentences= [
    "AI stands for Artificial intelligence",
    "Machine learning is the important subject",
    "Food is the main source of energy"
]
your_embeddings=model.encode(your_sentences)

#check similarity of my sentence
ysim_01=cosine_similarity(your_embeddings[0],your_embeddings[1])
ysim_02=cosine_similarity(your_embeddings[0],your_embeddings[2])
ysim_12=cosine_similarity(your_embeddings[1],your_embeddings[2])
print("Cosine similarities")
print()
print(f"Sentence 0 :{your_sentences[0]}")
print(f"Sentence 1 :{your_sentences[1]}")
print(f"Sentence 2 :{your_sentences[2]}")
print()
print(f"Similarity (0 vs 1): {ysim_01 :.4f} <- Expected: LOW (different topic)")
print(f"Similarity (0 vs 2): {ysim_02 :.4f} <- Expected: HIGH (same topic)")
print(f"Similarity (1 vs 2): {ysim_12 :.4f} <- Expected: LOW (different topic)")

Cosine similarities

Sentence 0 :AI stands for Artificial intelligence
Sentence 1 :Machine learning is the important subject
Sentence 2 :Food is the main source of energy

Similarity (0 vs 1): 0.3580 <- Expected: LOW (different topic)
Similarity (0 vs 2): 0.0639 <- Expected: HIGH (same topic)
Similarity (1 vs 2): 0.1995 <- Expected: LOW (different topic)


#UNIT 3 - ChromaDB - VECTOR DATABASE
--it is a vector database,

level 1: Simple English:

ChromDB is a filling cabinet for AI. Instead of filling by name or date, it files by meaning.Ask "give me files related to data cleaning" and it returns the closest ones

Level 2: Real-Life Analogy:

like music streaming app's recommendation engine - it finds songs similar to what you like, not songs with the same title

ChromaDB Key Terms
| Term | Meaning |
|:---|:---|
| **Collection** | like a table in SQL - a named group of documents |
| **Document** | The text of your record/note |
| **ID** | A unique identifier about the document (like a primary key) |
|**Meta Data** | Extra information about the document (e.g., subject,author,date)|
|**Distance** | How far apart two vectors are **lower=more similar**|


#CRICTICAL RULE : Distances vs Similarity

ChromaDB returns **distances**  -- NOT similarity scores


*   Distance **0.0**  = perfect match (identical vectors)
*   Distance **1.0**  = no match(completely different)
*   This is the **OPPOSITE** of cosine similarity score direction!



In [ ]:
#CELL 10
#STEP 1 -- Create a ChromaDB client and collection

#chromadb.Client() creates and IN-MEMORY database
#In-menory : data exists
#When you restart the kernel, data is gone and you must re-add it
#for permnent storage : use chromadb.persistentClient(path='./chroma_db')

chroma_client=chromadb.Client()

#get_or_create_collection: creates a new collection OR opens existing one
#'demo_notes' is the anme we give to this collection
#Think of it like : CREATE TABLE IF NOT EXISTS demo_notes

collection=chroma_client.get_or_create_collection(name='demo_notes')

print('chromaDB client created (in-memory mode)')
print(f"Collection name :demo_notes")
print(f"Documents in collection: {collection.count()}")

#count() returns how many documents are crrently in the collection
#should be 0 because we just created it

chromaDB client created (in-memory mode)
Collection name :demo_notes
Documents in collection: 0


In [ ]:
#STEP 2

#Let us add 5 sample documents
sample_docs=[
    "ETL stands for Extract Transform Load - the core data engineering process",
    "SQL SELECT statements retrieve data from database tables",
    "Machine learning models learns patters from training data",
    "Python Pandas library is used for datamanipulation and cleaning",
    "Neural networks are inspired by how the human brain works",
]

sample_ids=["doc_001","doc_002","doc_003","doc_004","doc_005"]
#IDs must be unique strings - like primary keys in SQL
#If you try to add the same ID twice, ChromaDB will raise an error

sample_metadata=[
    {"subject":"Data Engineering","topic":"ETL"},
    {"subject":"Data Engineering","topic":"SQL"},
    {"subject":"Data Science","topic":"Machine Learning"},
    {"subject":"Data Science","topic":"Pandas"},
    {"subject":"Data Science","topic":"Neural Networks"},
]

#Metadata : a list of dictionaries - one dict per document
#Each dict can have any keys you want
#Meta data is used for filtering later (eg only search ML document)

#Add all documents to the collection
collection.add(
    documents=sample_docs,       #The actual text content
    ids=sample_ids,              #Unique string IDs
    metadatas=sample_metadata    #Extra info about each document
)
print(f"Document added to collection")
print(f"Documents in collection: {collection.count()}")
#should show 5

Document added to collection
Documents in collection: 5


In [ ]:
#STEP 3  -  Query the collection by meaning

#IMPORTANT :Uses the SAME built-in model as collection.add()
#             - that is why same model consistency is critical

query="How do I clean and prepare data?"
#NOTICE: this query does not contain the words 'ETL' or 'Pandas'
#But it
results = collection.query(
    query_texts=query,
    n_results=3
)

print("RESULT KEYS AVAILABLE : ")
print(list(results.keys()))

RESULT KEYS AVAILABLE : 
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']


In [ ]:
#STEP 4 -  Display results in the Readable format

print(f"Query: '{query}'")
print("="*60)
print()

#results['documents'] is a list of lists
#the outer list has one element per query (we had 1 query)
#the inner list has one element per result
#so resuls['documents'][0] is the list of matched documents for out first query

matched_docs = results['documents'][0]          #list of matched texts
matched_ids = results['ids'][0]                 #list of matched ids
matched_distances = results['distances'][0]     #list of matched scores
matched_metadata = results['metadatas'][0]      #list of matched dicts

#zip() pairs elements from multiple lists together
#this is the same as doing matched_docs[0], matched_ids[0], etc in a loop

for rank, (doc, doc_id, dist, meta) in enumerate(zip(matched_docs, matched_ids, matched_distances, matched_metadata)):
  print(f"Rank: {rank} | ID: {doc_id} | Distance: {dist:.4f}")
  #Dictance: lower = more similar
  print(f" Subject:{meta['subject']}| Topic:{meta['topic']}")
  print(f" Document: {doc}")
  print()


print("NOTICE: The results are about ETL and Pandas - exactly what 'clean and prepare data' means")
print("Semantic search found them eveb through the words are different")

Query: 'How do I clean and prepare data?'

Rank: 0 | ID: doc_004 | Distance: 1.1117
 Subject:Data Science| Topic:Pandas
 Document: Python Pandas library is used for datamanipulation and cleaning

Rank: 1 | ID: doc_002 | Distance: 1.6139
 Subject:Data Engineering| Topic:SQL
 Document: SQL SELECT statements retrieve data from database tables

Rank: 2 | ID: doc_003 | Distance: 1.6720
 Subject:Data Science| Topic:Machine Learning
 Document: Machine learning models learns patters from training data

NOTICE: The results are about ETL and Pandas - exactly what 'clean and prepare data' means
Semantic search found them eveb through the words are different


In [ ]:
#Example : Find documents about learning, but ONLY from Machine Learning subject

filtered_query = "How do computers learn from examples?"

filtered_results = collection.query(
    query_texts=[filtered_query],
    n_results=3,
    where={"subject": "Machine Learning"}
    #Where: a dictionary that filters
)

print(f"FILTERED QUERY: '{filtered_query}'")
print("Filter: Only Machine Learning documents")
print("="*60)

for rank, (doc, dist, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
    filtered_results['metadatas'][0]
), start=1):
    print(f"Rank: {rank} | Distance: {dist:.4f}| Subject:{meta['subject']}")
    print(f" {doc}")
    print()



print("NOTICE: Only ML documents appear,even through ETL and Pandas")

FILTERED QUERY: 'How do computers learn from examples?'
Filter: Only Machine Learning documents
NOTICE: Only ML documents appear,even through ETL and Pandas


In [ ]:
#if you want the similarity score : similarity =1 - distance

print("DISTANCE TO SIILARITY CONVERSION")
print("="*50)
print(f"{'Distance':<15}{'Similarity':<15}{'Iterpretation':<20}")
print("-"*50)

distances = [0.05, 0.20, 0.40, 0.65, 0.90]
interpretations =["Near identical","Very similar","Related","Somewhat related","Not Related"]

for dist, interp in zip(distances, interpretations):
  similarity=1-dist   # convert distance to similarity score
  print(f"{dist:<15.2f} {similarity:<15.2f} {interp:<20}")

DISTANCE TO SIILARITY CONVERSION
Distance       Similarity     Iterpretation       
--------------------------------------------------
0.05            0.95            Near identical      
0.20            0.80            Very similar        
0.40            0.60            Related             
0.65            0.35            Somewhat related    
0.90            0.10            Not Related         


#UNIT 4 -- Load the college notes dataset

Now we will work with the real dataset for today

In [ ]:
#Option 1: Upload
#Option 2: if you have it in your google drive, mount drive first

#for taday's session, the file should be in the same folder as this notebook
notes_df=pd.read_csv('college_notes.csv')

#notes_df is now a DataFrame  - a table with rows and columns

print("Dataset loaded successfully!")
print(f"Shape: {notes_df.shape}")
print(f"First 5 rows:")
notes_df.head()


Dataset loaded successfully!
Shape: (15, 4)
First 5 rows:


,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [ ]:
#how many notes are in each subject?
#value_counts() count how many times each unique value appears

print("Notes per subject : ")
print(notes_df['subject'].value_counts())
#subject is the column name



Notes per subject : 
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64


In [ ]:
#iloc[0] selects the first row by position (index 0)


first_note = notes_df.iloc[0]
print(f"Note ID: {first_note['note_id']}")
print(f"Subject: {first_note['subject']}")
print(f"Topic: {first_note['content']}")
print(f"Content: {first_note['topic']}")

Note ID: N001
Subject: Data Engineering
Topic: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
Content: ETL Pipelines


In [ ]:
all_documents=notes_df['content'].tolist()
#all_documents is a list of 15 strings - the text content of each note

all_ids=notes_df['note_id'].tolist()
#all_ids is a list of 15 strings  -  N001, N002, ... ,N015

all_metadatas=notes_df.to_dict('records')

#MINI PROJECT  -- Smart Notes Search Engine
Now we combine everything to build a working semantic search system.

**Project Overview**


**Goal** : build a search engine that finds relevant college notes by MEANING , not just keywords

**Steps**:
1. create a new chromaDB collection for college notes
2. index all 15 notes in the collection
3. run semantic queries and display top result
4. filter results by subjects
5.

In [3]:
!pip install chromadb sentence-transformers pandas -q
print("Successfully installed")

In [4]:
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

In [5]:
df = pd.read_csv("college_notes.csv")

df.head()

,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [6]:
print(df.shape)
print(df.columns)

(15, 4)
Index(['note_id', 'subject', 'topic', 'content'], dtype='object')


In [7]:
#create chromodb collection
client = chromadb.Client()

collection = client.create_collection(
    name="college_notes"
)

print("Collection Created")

Collection Created


In [8]:
#index all notes
collection.add(
    ids=df["note_id"].astype(str).tolist(),
    documents=df["content"].tolist(),
    metadatas=[
        {
            "subject": row["subject"],
            "topic": row["topic"]
        }
        for _, row in df.iterrows()
    ]
)

print("All notes indexed successfully")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 44.3MiB/s]


All notes indexed successfully


In [9]:
print("Total Notes:", collection.count())

Total Notes: 15


In [10]:
#sementic search
query = "How do we collect data from websites and applications?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i, doc in enumerate(results["documents"][0],1):
    print(f"{i}. {doc}")
    print()

1. An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.

2. A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.

3. Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is like a table with rows and columns. Common operations include reading CSV files filtering rows grouping data and creating new columns.



In [11]:
#Display Topic and Subject
query = "How do we collect data from websites and applications?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i in range(len(results["documents"][0])):

    print("Subject:",
          results["metadatas"][0][i]["subject"])

    print("Topic:",
          results["metadatas"][0][i]["topic"])

    print("Content:")
    print(results["documents"][0][i])

    print("-"*60)

Subject: Data Engineering
Topic: APIs and Data Collection
Content:
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.
------------------------------------------------------------
Subject: Data Engineering
Topic: SQL Databases
Content:
A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.
------------------------------------------------------------
Subject: Python Programming
Topic: Pandas Library
Content:
Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is like a table with rows and columns. Common operations include reading CSV files filtering rows grouping data and creating new columns.
--------

In [12]:
query = "How can machine learning understand language?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for doc in results["documents"][0]:
    print(doc)
    print()

A Large Language Model or LLM is an AI model trained on massive amounts of text data. It can generate human-like text answer questions summarize documents and perform many language tasks. Examples include GPT Claude and LLaMA.

Supervised learning is a type of machine learning where the model learns from labeled data. The model is given input features and correct output labels and it learns to predict outputs for new unseen inputs. Examples include classification and regression.

Model evaluation measures how well a machine learning model performs. Common metrics include accuracy for classification and Mean Absolute Error and R-squared for regression. A good model generalizes well to new data it has not seen before.



In [13]:
#filter by subject
query = "How do chatbots generate responses?"

results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"subject":"Artificial Intelligence"}
)

for doc in results["documents"][0]:
    print(doc)
    print()

In [14]:
#keyword search function
def keyword_search(query):

    query = query.lower()

    matches = []

    for _, row in df.iterrows():

        if query in row["content"].lower():

            matches.append({
                "topic": row["topic"],
                "content": row["content"]
            })

    return matches

In [15]:
#Compare Keyword vs Semantic Search
query = "systems that learn patterns from data"

In [16]:
print("KEYWORD SEARCH")
print("="*50)

keyword_results = keyword_search(query)

print(keyword_results)

KEYWORD SEARCH
[]


In [18]:
print("SEMANTIC SEARCH")
print("="*50)

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i in range(len(results["documents"][0])):

    print("Topic:",
          results["metadatas"][0][i]["topic"])

    print()

SEMANTIC SEARCH
Topic: Supervised Learning

Topic: Decision Trees

Topic: Data Visualization



In [19]:
#create results dataframe
query = "database management"

results = collection.query(
    query_texts=[query],
    n_results=5
)

output = pd.DataFrame({
    "Topic":[m["topic"] for m in results["metadatas"][0]],
    "Subject":[m["subject"] for m in results["metadatas"][0]],
    "Content":results["documents"][0]
})

output

,Topic,Subject,Content
0,SQL Databases,Data Engineering,A database is an organized collection of data ...
1,Data Cleaning,Data Engineering,Data cleaning involves fixing or removing inco...
2,Supervised Learning,Machine Learning,Supervised learning is a type of machine learn...
3,Decision Trees,Machine Learning,A decision tree is a machine learning model th...
4,Large Language Models,Generative AI,A Large Language Model or LLM is an AI model t...


#PRACTICE QUESTIONS -- DAY 7

answer these in the cells below. they cover all three units
1. Q1(Conceptual) :What is the difference between keywords search and semantic search? give one real-world example for each

2. Q2(Technical) : What does ```model.encode(['hello world'])``` return?What is the shape of the output?

3. Q3(critical thinking): A student adds document using model A but queries using model B. will results be correct? why or why not?

4. Q4(Application) : ChromaDB retuns distance [0.12, 0.45, 0.87].Which is most relevant? Which is the least relevant?

5. Q5(Code) : write the ```collection.add()``` call to store

### Q1 (Conceptual): What is the difference between keyword search and semantic search? Give one real-world example for each.

**Keyword Search**: Matches documents based on the presence of exact words or phrases in the query. It's good for precise, literal matches.
*   **Example**: Searching for "red shoes" on an e-commerce site will only show products with the exact phrase "red shoes" in their description.

**Semantic Search**: Understands the meaning and context of the query, returning results that are conceptually related, even if they don't contain the exact keywords. It uses embeddings to capture meaning.
*   **Example**: Searching for "comfortable footwear for running" might return results for "running sneakers" or "athletic shoes" even if the word "footwear" isn't present.

### Q2 (Technical): What does `model.encode(['hello world'])` return? What is the shape of the output?

`model.encode(['hello world'])` returns a NumPy array containing the vector embedding (a list of numbers) for the input sentence "hello world".

**Shape of the output**: `(1, 384)`
*   `1` represents the number of sentences encoded (in this case, one sentence).
*   `384` represents the dimensionality of the embedding vector produced by the `all-MiniLM-L6-v2` model.

### Q3 (Critical Thinking): A student adds documents using model A but queries using model B. Will results be correct? Why or why not?

No, the results will likely **not be correct** or at least suboptimal. Here's why:

*   **Incompatible Vector Spaces**: Each embedding model (Model A and Model B) creates its own unique vector space where it places sentences based on its learned understanding of meaning. If documents are embedded using Model A, they reside in Model A's vector space. If queries are embedded using Model B, they are in Model B's vector space.
*   **Meaningful Comparison Fails**: Comparing a vector from Model A's space with a vector from Model B's space using distance metrics (like cosine similarity) is like comparing apples and oranges. The distances or similarities calculated will not accurately reflect the semantic relationship between the query and the documents because their numerical representations are not aligned.

For accurate semantic search, the same embedding model **must be used** for both indexing (adding documents) and querying.

### Q5 (Code): Write the `collection.add()` call to store your `df` DataFrame into a ChromaDB collection named `my_notes`, with `note_id` as IDs, `content` as documents, and both `subject` and `topic` as metadata.

In [20]:
# Assuming 'df' is already loaded and contains 'note_id', 'subject', 'topic', and 'content' columns

import chromadb

# Create a new client (or use an existing one)
client_q5 = chromadb.Client() # For in-memory, or chromadb.PersistentClient(path='./my_db') for persistent

# Get or create the collection
collection_q5 = client_q5.get_or_create_collection(name="my_notes")

# Add documents to the collection
collection_q5.add(
    ids=df["note_id"].astype(str).tolist(), # IDs must be strings
    documents=df["content"].tolist(),
    metadatas=[
        {"subject": row["subject"], "topic": row["topic"]}
        for index, row in df.iterrows()
    ]
)

print(f"Added {collection_q5.count()} documents to 'my_notes' collection.")

Added 15 documents to 'my_notes' collection.
